# Import Statements

In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import os
import wandb

In [2]:
# os.environ["WANDB_DISABLED"] = "true"
wandb.login()

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

# Train Dataset

In [3]:
df = pd.read_csv('../../data/train.csv')
df.head()

,id,text,anger,fear,joy,sadness,surprise,emotions
0,0,the dentist that did the work apparently did a...,1,0,0,1,0,['anger' 'sadness']
1,1,i'm gonna absolutely ~~suck~~ be terrible duri...,0,1,0,1,0,['fear' 'sadness']
2,2,"bridge: so leave me drowning calling houston, ...",0,1,0,1,0,['fear' 'sadness']
3,3,after that mess i went to see my now ex-girlfr...,1,1,0,1,0,['anger' 'fear' 'sadness']
4,4,"as he stumbled i ran off, afraid it might some...",0,1,0,0,0,['fear']


In [4]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']
for col in emotion_cols:
    df[col] = df[col].astype(int)

# Set Device

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
labels = emotion_cols
id2label = {idx: label for idx, label in enumerate(labels)}
label2id = {label: idx for idx, label in enumerate(labels)}

# Set Model

In [7]:
model_name = "d_bert_train"

# Emotion Dataset Class

In [8]:
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)

        # Tokenize the text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': labels
        }

In [9]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
X_train = train_df['text'].tolist()
y_train = train_df[labels].values.tolist()
X_val = val_df['text'].tolist()
y_val = val_df[labels].values.tolist()

# F1 Score

In [10]:
def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    # Apply sigmoid and threshold
    sigmoid_preds = torch.sigmoid(torch.Tensor(preds))
    binary_preds = (sigmoid_preds > 0.5).int().numpy()
    
    true_labels = p.label_ids

    f1_macro = f1_score(y_true=true_labels, y_pred=binary_preds, average='macro', zero_division=0)
    
    return {
        'f1': f1_macro
    }

# Select Checkpoint

In [11]:
MODEL_CHECKPOINT = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

In [12]:
train_dataset = EmotionDataset(X_train, y_train, tokenizer)
val_dataset = EmotionDataset(X_val, y_val, tokenizer)

# Initialize Model

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)
model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


# Initialize Arguments

In [14]:
# Define training arguments
training_args = TrainingArguments(
    output_dir=f'./results/{model_name}',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name=f"{model_name}-run",
    save_total_limit=1
)

# Initialize WandB

In [15]:
wandb.init(
    project="24f1002325-t32025", 
    entity="24f1002325-iit-madras",
    name=f"{model_name}-lr{2e-5}-epochs{3}",
    config={
        "architecture": model_name,
        "learning_rate": 2e-5,
        "epochs": 3,
        "batch_size": 16
    },
    reinit=True
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


# Model Training

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\91762\AppData\Local\Temp\ipykernel_11308\4030190838.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,0.420600,0.371461,0.657017
2,0.311800,0.327990,0.728186
3,0.256400,0.315101,0.735017


TrainOutput(global_step=1026, training_loss=0.34864104839793425, metrics={'train_runtime': 314.6803, 'train_samples_per_second': 52.062, 'train_steps_per_second': 3.26, 'total_flos': 542582375051520.0, 'train_loss': 0.34864104839793425, 'epoch': 3.0})

# Save Model

In [17]:
save_path = f"../../models/{model_name}"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model and tokenizer for {model_name} saved to {save_path}")
wandb.finish()

Model and tokenizer for d_bert_train saved to ../../models/d_bert_train


eval/f1,▁▇█
eval/loss,█▃▁
eval/runtime,▁▇█
eval/samples_per_second,█▂▁
eval/steps_per_second,█▂▁
train/epoch,▁▂▃▃▃▄▅▅▆▆▇███
train/global_step,▁▂▃▃▃▄▅▅▆▆▇███
train/grad_norm,▁▂▁▁▇█▃▃█▄
train/learning_rate,█▇▆▆▅▄▃▃▂▁
train/loss,█▆▅▃▃▂▂▁▁▁
eval/f1,0.73502
